# Getting Started with Foundry Agent Service

Build your first AI agent with **Microsoft Foundry Agent Service** — the production-ready runtime for building, deploying, and managing AI agents at scale.

## What you'll learn

1. What Foundry Agent Service is and why it matters
2. How to set up your environment and permissions
3. Creating your first agent with the Python SDK
4. Chatting with your agent using the Conversations API
5. Adding tools: **Code Interpreter**, **File Search**, and **Web Search**
6. Building a multi-tool agent
7. Publishing your agent to Microsoft 365 Copilot & Teams

## Prerequisites

- An Azure subscription ([create one free](https://azure.microsoft.com/pricing/purchase-options/azure-account))
- Python 3.10+
- Azure CLI installed and signed in (`az login`)

---

## 1. What is Foundry Agent Service?

Announced at **Microsoft Ignite 2025**, Microsoft Foundry (formerly Azure AI Studio) is the unified platform for building AI applications. At the heart of Foundry is **Agent Service** — a managed runtime that connects models, tools, and frameworks into a single, production-ready system.

Agent Service handles the hard parts for you:

| Capability | What it does |
|---|---|
| **Conversation management** | Maintains multi-turn state across sessions |
| **Tool orchestration** | Routes to Code Interpreter, File Search, Bing, custom functions, MCP servers, and more |
| **Content safety** | Enforces responsible AI guardrails by default |
| **Identity & RBAC** | Integrates with Microsoft Entra ID for secure, least-privilege access |
| **Observability** | Built-in tracing with Application Insights and OpenTelemetry |
| **Scale** | Auto-scales from prototype to production without infrastructure changes |

### Key concepts

An agent has three core components:

- **Model (LLM)** — Powers reasoning and language understanding (e.g., GPT-4.1, GPT-5)
- **Instructions** — Define the agent's goals, behavior, and constraints
- **Tools** — Extend the agent's capabilities (code execution, search, custom functions, etc.)

### Architecture at a glance

```
┌─────────────────────────────────────────────────────┐
│                  Your Application                    │
│   AIProjectClient  →  OpenAI Client (Responses API) │
└──────────────┬──────────────────────┬───────────────┘
               │                      │
     ┌─────────▼─────────┐  ┌────────▼────────┐
     │  Agent Definitions │  │  Conversations   │
     │  (PromptAgent /    │  │  (Multi-turn     │
     │   WorkflowAgent)   │  │   state mgmt)    │
     └─────────┬─────────┘  └────────┬────────┘
               │                      │
     ┌─────────▼──────────────────────▼───────┐
     │        Foundry Agent Service Runtime    │
     │  ┌──────────┐ ┌────────┐ ┌───────────┐ │
     │  │Code Interp│ │File    │ │Bing/Web   │ │
     │  │          │ │Search  │ │Search     │ │
     │  └──────────┘ └────────┘ └───────────┘ │
     │  ┌──────────┐ ┌────────┐ ┌───────────┐ │
     │  │Functions │ │MCP     │ │SharePoint │ │
     │  │          │ │Servers │ │           │ │
     │  └──────────┘ └────────┘ └───────────┘ │
     └────────────────────────────────────────┘
```

---

## 2. Set Up Your Environment

### Required RBAC permissions

Before you begin, make sure you have the right roles assigned in Azure:

| Action | Required Role |
|---|---|
| Create a Foundry account and project | **Azure AI Account Owner** |
| Create and edit agents (SDK / Playground) | **Azure AI User** |
| Deploy models | **Azure AI Project Manager** |
| Assign RBAC for Standard setup resources | **Role Based Access Control Administrator** |

> **Tip:** For most developers getting started, the **Azure AI User** role on the project scope is sufficient. An admin with **Azure AI Account Owner** can set up the account and project, then assign **Azure AI User** to team members.

### Install the SDK

The new Foundry SDK v2 uses two clients:
- `AIProjectClient` — for managing agents, connections, and project resources
- `OpenAI client` (from project) — for conversations and responses

In [ ]:
%pip install "azure-ai-projects>=2.0.0b3" azure-identity python-dotenv openai --quiet

### Configure environment variables

Create a `.env` file in your project root with the following values:

```env
PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
MODEL_NAME=gpt-5.2
```

You can find your **Project Endpoint** in the _new_ [Foundry portal](https://ai.azure.com/nextgen) under your project's overview page.

> **Note:** This tutorial uses `gpt-5.2` as the model deployment name. Substitute with your own deployed model (e.g., `gpt-4.1`, `gpt-4o-mini`) based on your available [quotas and limits](https://learn.microsoft.com/azure/ai-foundry/agents/quotas-limits).

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

# Initialize the project client
project_client = AIProjectClient(
    endpoint=os.environ["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

# Get the OpenAI client from the project
openai_client = project_client.get_openai_client()

MODEL = os.environ.get("MODEL_NAME", "gpt-5.2")
print(f"✅ Connected to project: {os.environ['PROJECT_ENDPOINT']}")
print(f"✅ Using model: {MODEL}")

**Expected output:**
```
✅ Connected to project: https://<your-resource>.services.ai.azure.com/api/projects/<your-project>
✅ Using model: gpt-5.2
```

---

## 3. Create Your First Agent

An agent is defined using a `PromptAgentDefinition` — a declarative specification that combines the model, instructions, and tools. The agent is published as a versioned resource using `agents.create_version()`.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

# Create a simple conversational agent
agent = project_client.agents.create_version(
    agent_name="my-first-agent",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a helpful AI assistant. "
            "Be concise, accurate, and friendly. "
            "If you don't know something, say so honestly."
        ),
    ),
)

print(f"✅ Agent created")
print(f"   Name:    {agent.name}")
print(f"   Version: {agent.version}")
print(f"   ID:      {agent.id}")

**Expected output:**
```
✅ Agent created
   Name:    my-first-agent
   Version: 1
   ID:      asst_abc123xyz
```

---

## 4. Chat with Your Agent

The SDK v2 uses the **Responses API** with optional **Conversations** for multi-turn state management. This follows the same patterns as the OpenAI API but runs against your Foundry project.

### Single-turn response

In [ ]:
# Send a single message to your agent
response = openai_client.responses.create(
    extra_body={"agent": {"name": "my-first-agent", "type": "agent_reference"}},
    input="What are the three laws of robotics?",
)

print(response.output_text)

**Expected output:**
```
The Three Laws of Robotics, formulated by Isaac Asimov, are:

1. **First Law:** A robot may not injure a human being or, through inaction,
   allow a human being to come to harm.
2. **Second Law:** A robot must obey orders given it by human beings, except
   where such orders would conflict with the First Law.
3. **Third Law:** A robot must protect its own existence as long as such
   protection does not conflict with the First or Second Law.
```

### Multi-turn conversation

Use a **Conversation** to maintain context across multiple exchanges:

In [ ]:
# Create a conversation for multi-turn context
conversation = openai_client.conversations.create()
print(f"📝 Conversation ID: {conversation.id}\n")

# First turn
response1 = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent": {"name": "my-first-agent", "type": "agent_reference"}},
    input="What is the capital of France?",
)
print(f"User: What is the capital of France?")
print(f"Agent: {response1.output_text}\n")

# Follow-up (agent remembers context)
response2 = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent": {"name": "my-first-agent", "type": "agent_reference"}},
    input="What is its population?",
)
print(f"User: What is its population?")
print(f"Agent: {response2.output_text}")

**Expected output:**
```
📝 Conversation ID: conv_abc123

User: What is the capital of France?
Agent: The capital of France is Paris.

User: What is its population?
Agent: Paris has a population of approximately 2.1 million people in the city
proper, and about 12.4 million in the greater metropolitan area.
```

---

## 5. Add Tools to Your Agent

Tools extend what your agent can do beyond text generation. Foundry Agent Service supports a rich catalog of built-in tools:

| Tool | What it does |
|---|---|
| **Code Interpreter** | Executes Python code in a sandboxed environment |
| **File Search** | Searches over uploaded documents using vector retrieval |
| **Bing Grounding** | Grounds responses with real-time web search results |
| **Function Calling** | Calls your custom functions |
| **MCP Servers** | Connects agents to Model Context Protocol tool servers |
| **SharePoint** | Searches SharePoint sites and documents |
| **Microsoft Fabric** | Queries Fabric data agents |
| **Image Generation** | Generates images from text prompts |
| **A2A (Agent-to-Agent)** | Connects agents that support the A2A protocol |

Let's explore the three most commonly used tools.

### 5a. Code Interpreter

The Code Interpreter tool gives your agent the ability to write and execute Python code in a secure, sandboxed container. It can:
- Perform calculations and data analysis
- Generate charts and visualizations
- Process and transform files

In [ ]:
from openai.types.responses import ResponseTool

# Create an agent with Code Interpreter
code_agent = project_client.agents.create_version(
    agent_name="code-interpreter-agent",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a data analysis assistant. "
            "Use the code interpreter to run Python code when asked "
            "to perform calculations or create visualizations."
        ),
        tools=[ResponseTool.create_code_interpreter_tool()],
    ),
)
print(f"✅ Code Interpreter agent created (version: {code_agent.version})")

In [ ]:
# Ask the agent to perform a calculation
response = openai_client.responses.create(
    extra_body={"agent": {"name": "code-interpreter-agent", "type": "agent_reference"}},
    input="Calculate the first 20 Fibonacci numbers and show them as a list.",
)

print(response.output_text)

**Expected output:**
```
Here are the first 20 Fibonacci numbers:

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]
```

The agent wrote and executed Python code behind the scenes to compute the result!

### 5b. File Search

The File Search tool lets your agent search over your uploaded documents using semantic vector retrieval. It automatically indexes your files and can answer questions grounded in their content.

#### How it works
1. Upload files to a **Vector Store** in your project
2. Attach the vector store to an agent via `FileSearchTool`
3. The agent automatically searches the indexed documents when relevant

In [ ]:
from openai.types.responses import ResponseTool

# Create an agent with File Search
# Note: In production, you would first upload files to a vector store
# and reference the vector_store_id in the tool configuration.
#
# Example of uploading files (run separately):
# vector_store = openai_client.vector_stores.create(name="my-docs")
# openai_client.vector_stores.files.upload(
#     vector_store_id=vector_store.id,
#     file=open("document.pdf", "rb")
# )

file_search_agent = project_client.agents.create_version(
    agent_name="file-search-agent",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a research assistant. "
            "Search through the uploaded documents to answer questions. "
            "Always cite the source document when providing information."
        ),
        tools=[ResponseTool.create_file_search_tool()],
    ),
)
print(f"✅ File Search agent created (version: {file_search_agent.version})")
print("\n💡 To use this agent, upload documents to a vector store first.")
print("   See: https://learn.microsoft.com/azure/ai-foundry/agents/how-to/tools/file-search")

**Expected output:**
```
✅ File Search agent created (version: 1)

💡 To use this agent, upload documents to a vector store first.
   See: https://learn.microsoft.com/azure/ai-foundry/agents/how-to/tools/file-search
```

### 5c. Web Search (Bing Grounding)

The Bing Grounding tool lets your agent search the web for real-time information. It requires a **Bing Search** resource connected to your Foundry project.

#### Prerequisites for Web Search
1. Create a **Bing Search** resource in the Azure portal
2. Add a **project connection** for the Bing resource in the Foundry portal
3. Note the connection name for use in the SDK

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition, BingGroundingTool

# To use Bing Grounding, you need a Bing connection in your project.
# Uncomment and set the connection name:
#
# BING_CONNECTION_NAME = "my-bing-connection"
# bing_connection = project_client.connections.get(BING_CONNECTION_NAME)
#
# web_search_agent = project_client.agents.create_version(
#     agent_name="web-search-agent",
#     definition=PromptAgentDefinition(
#         model=MODEL,
#         instructions=(
#             "You are a research assistant with access to web search. "
#             "Use Bing to find current information when needed. "
#             "Always cite your sources."
#         ),
#         tools=[BingGroundingTool(project_connection_id=bing_connection.id)],
#     ),
# )
# print(f"✅ Web Search agent created (version: {web_search_agent.version})")

print("ℹ️  Web Search requires a Bing Search connection in your Foundry project.")
print("   1. Create a Bing Search resource in the Azure portal")
print("   2. Add a project connection in the Foundry portal")
print("   3. Uncomment the code above and set BING_CONNECTION_NAME")
print("   See: https://learn.microsoft.com/azure/ai-foundry/agents/how-to/tools/bing-tools")

**Expected output:**
```
ℹ️  Web Search requires a Bing Search connection in your Foundry project.
   1. Create a Bing Search resource in the Azure portal
   2. Add a project connection in the Foundry portal
   3. Uncomment the code above and set BING_CONNECTION_NAME
   See: https://learn.microsoft.com/azure/ai-foundry/agents/how-to/tools/bing-tools
```

---

## 6. Build a Multi-Tool Agent

The real power of Foundry Agent Service comes from combining multiple tools in a single agent. The agent's reasoning layer automatically decides which tool to use based on the user's request.

In [ ]:
from openai.types.responses import ResponseTool

# Create a multi-tool agent with Code Interpreter + File Search
multi_tool_agent = project_client.agents.create_version(
    agent_name="multi-tool-agent",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a versatile AI assistant with multiple capabilities:\n"
            "- Use Code Interpreter for calculations, data analysis, and visualizations\n"
            "- Use File Search to find information in uploaded documents\n\n"
            "Choose the right tool for each task. Be thorough and cite sources."
        ),
        tools=[
            ResponseTool.create_code_interpreter_tool(),
            ResponseTool.create_file_search_tool(),
        ],
    ),
)

print(f"✅ Multi-tool agent created")
print(f"   Name:    {multi_tool_agent.name}")
print(f"   Version: {multi_tool_agent.version}")
print(f"   Tools:   Code Interpreter, File Search")

In [ ]:
# Test the multi-tool agent with a calculation task
response = openai_client.responses.create(
    extra_body={"agent": {"name": "multi-tool-agent", "type": "agent_reference"}},
    input=(
        "Calculate the compound interest on a $10,000 investment "
        "at 7% annual rate over 10 years, compounded monthly. "
        "Show the formula and the final amount."
    ),
)

print(response.output_text)

**Expected output:**
```
The compound interest formula is:

A = P(1 + r/n)^(nt)

Where:
- P = $10,000 (principal)
- r = 0.07 (annual interest rate)
- n = 12 (compounded monthly)
- t = 10 (years)

A = 10000 × (1 + 0.07/12)^(12×10)
A = 10000 × (1.005833...)^120
A ≈ $20,096.61

After 10 years, your $10,000 investment would grow to approximately
$20,096.61, earning about $10,096.61 in compound interest.
```

---

## 7. Streaming Responses

For a better user experience, especially with longer responses, you can stream the output token-by-token:

In [ ]:
# Stream a response from the agent
stream = openai_client.responses.create(
    extra_body={"agent": {"name": "my-first-agent", "type": "agent_reference"}},
    input="Write a haiku about building AI agents.",
    stream=True,
)

print("Agent (streaming): ", end="")
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()  # newline at the end

**Expected output:**
```
Agent (streaming): Code and intent merge
Tools extend the agent's reach
Answers come to life
```

---

## 8. Clean Up Resources

Delete agent versions you no longer need to keep your project organized:

In [ ]:
# Clean up agents created in this tutorial
agents_to_clean = [
    ("my-first-agent", agent.version),
    ("code-interpreter-agent", code_agent.version),
    ("file-search-agent", file_search_agent.version),
    ("multi-tool-agent", multi_tool_agent.version),
]

for name, version in agents_to_clean:
    try:
        project_client.agents.delete_version(agent_name=name, version=version)
        print(f"🗑️  Deleted: {name} v{version}")
    except Exception as e:
        print(f"⚠️  Could not delete {name} v{version}: {e}")

print("\n✅ Cleanup complete")

**Expected output:**
```
🗑️  Deleted: my-first-agent v1
🗑️  Deleted: code-interpreter-agent v1
🗑️  Deleted: file-search-agent v1
🗑️  Deleted: multi-tool-agent v1

✅ Cleanup complete
```

---

## 9. Publish to Microsoft 365 Copilot & Teams

Once your agent is working well, you can publish it so others in your organization can use it directly in Microsoft 365 Copilot and Microsoft Teams.

### Publishing workflow

```
┌──────────────┐    ┌──────────────────┐    ┌─────────────────┐
│ Build & Test  │───▶│ Publish as Agent  │───▶│ Available in    │
│ in Foundry    │    │ Application       │    │ M365 Copilot    │
│               │    │                   │    │ & Teams         │
└──────────────┘    └──────────────────┘    └─────────────────┘
```

### Steps to publish

1. **Test thoroughly** in the Foundry portal or playground
2. **Publish** your agent as an Agent Application:
   - In the Foundry portal, select your agent version
   - Click **Publish** → **Publish to Teams and Microsoft 365 Copilot**
   - An Azure Bot Service resource is created automatically
3. **Configure metadata** (name, description, icons, privacy policy)
4. **Choose distribution scope**:

| Scope | Visibility | Admin Approval | Best for |
|---|---|---|---|
| **Shared** | Your agents list | Not required | Personal testing, small teams |
| **Organization** | Built by your org | Required | Organization-wide deployment |

5. **Download the package** and upload to Teams for testing
6. For organization scope: request **admin approval** in the M365 admin center

### Required roles for publishing

| Action | Required Role |
|---|---|
| Publish agents | **Azure AI Project Manager** on Foundry project |
| Invoke/chat with published agents | **Azure AI User** on Agent Application |
| Approve org-wide distribution | **Microsoft 365 Admin** |

### Programmatic publishing (REST API)

You can also publish via the REST API:

```bash
# 1. Create an Agent Application
PUT https://management.azure.com/subscriptions/{sub}/resourceGroups/{rg}/\
    providers/Microsoft.CognitiveServices/accounts/{account}/\
    projects/{project}/applications/{app_name}?api-version={version}

# 2. Create a Deployment
PUT https://management.azure.com/subscriptions/{sub}/resourceGroups/{rg}/\
    providers/Microsoft.CognitiveServices/accounts/{account}/\
    projects/{project}/applications/{app_name}/\
    agentdeployments/{deployment_name}?api-version={version}
```

> **Important:** A published agent gets its own **Agent Identity** in Microsoft Entra ID, separate from your project identity. Make sure to reassign RBAC permissions to the new identity for any Azure resources the agent accesses.

📚 Full guide: [Publish agents to Microsoft 365 Copilot and Teams](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/publish-copilot)

---

## Next Steps

You've built your first agent, added tools, and learned how to publish to Microsoft 365. Here's where to go next:

| Topic | Link |
|---|---|
| **Function Calling** | Define custom functions your agent can invoke |
| **Multi-Agent Workflows** | Orchestrate multiple agents with `WorkflowAgentDefinition` |
| **MCP Integration** | Connect agents to Model Context Protocol servers |
| **Hosted Agents** | Deploy containerized agent code to Foundry |
| **Tracing & Monitoring** | Set up Application Insights for observability |
| **Evaluation** | Measure agent quality with built-in evaluators |

### Useful links

- [Foundry Agent Service overview](https://learn.microsoft.com/azure/ai-foundry/agents/overview)
- [Environment setup guide](https://learn.microsoft.com/azure/ai-foundry/agents/environment-setup)
- [RBAC for Microsoft Foundry](https://learn.microsoft.com/azure/ai-foundry/concepts/rbac-foundry)
- [Official Python samples](https://github.com/azure-ai-foundry/foundry-samples/tree/main/samples/python)
- [Azure AI Projects SDK reference](https://learn.microsoft.com/python/api/azure-ai-projects)